In [1]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded!")

Libraries loaded!


In [2]:
# Load all 10 datasets
# converting the CSVs to datframes
fund_master = pd.read_csv('../data/raw/01_fund_master.csv')
nav_history = pd.read_csv('../data/raw/02_nav_history.csv')
aum_by_house = pd.read_csv('../data/raw/03_aum_by_fund_house.csv')
monthly_sip = pd.read_csv('../data/raw/04_monthly_sip_inflows.csv')
category_inflows = pd.read_csv('../data/raw/05_category_inflows.csv')
folio_count = pd.read_csv('../data/raw/06_industry_folio_count.csv')
scheme_perf = pd.read_csv('../data/raw/07_scheme_performance.csv')
transactions = pd.read_csv('../data/raw/08_investor_transactions.csv')
portfolio = pd.read_csv('../data/raw/09_portfolio_holdings.csv')
benchmark = pd.read_csv('../data/raw/10_benchmark_indices.csv')

# putting all of them in one dictionary

datasets = {
    'fund_master': fund_master,
    'nav_history': nav_history,
    'aum_by_house': aum_by_house,
    'monthly_sip': monthly_sip,
    'category_inflows': category_inflows,
    'folio_count': folio_count,
    'scheme_perf': scheme_perf,
    'transactions': transactions,
    'portfolio': portfolio,
    'benchmark': benchmark
}

print("All datasets loaded!\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns | Missing: {df.isnull().sum().sum()} | Duplicates: {df.duplicated().sum()}")

All datasets loaded!

fund_master: 40 rows, 15 columns | Missing: 0 | Duplicates: 0
nav_history: 46000 rows, 3 columns | Missing: 0 | Duplicates: 0
aum_by_house: 90 rows, 5 columns | Missing: 0 | Duplicates: 0
monthly_sip: 48 rows, 6 columns | Missing: 12 | Duplicates: 0
category_inflows: 144 rows, 3 columns | Missing: 0 | Duplicates: 0
folio_count: 21 rows, 6 columns | Missing: 0 | Duplicates: 0
scheme_perf: 40 rows, 19 columns | Missing: 0 | Duplicates: 0
transactions: 32778 rows, 13 columns | Missing: 0 | Duplicates: 0
portfolio: 322 rows, 8 columns | Missing: 0 | Duplicates: 0
benchmark: 8050 rows, 3 columns | Missing: 0 | Duplicates: 0


In [5]:
print("Monthly SIP missing values:")
print(monthly_sip.isnull().sum())
print("\nSample of rows with missing values:")
print(monthly_sip[monthly_sip.isnull().any(axis=1)])

Monthly SIP missing values:
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

Sample of rows with missing values:
      month  sip_inflow_crore  active_sip_accounts_crore  \
0   2022-01             11517                       4.91   
1   2022-02             11438                       4.93   
2   2022-03             12328                       5.09   
3   2022-04             11863                       5.48   
4   2022-05             12286                       5.55   
5   2022-06             12276                       5.60   
6   2022-07             12140                       5.65   
7   2022-08             12694                       5.71   
8   2022-09             12976                       5.80   
9   2022-10             13040                       5.93   
10  2022-11             13306                       6.00   
11  2022-1

In [6]:
# ============================================
# CLEAN 02_nav_history.csv
# ============================================

print("BEFORE CLEANING:")
print("Shape:", nav_history.shape)
print("\nDtypes:")
print(nav_history.dtypes)
print("\nSample:")
print(nav_history.head())
print("\nNAV <= 0:", (nav_history['nav'] <= 0).sum())
print("Duplicates:", nav_history.duplicated().sum())
print("Missing:", nav_history.isnull().sum().sum())

BEFORE CLEANING:
Shape: (46000, 3)

Dtypes:
amfi_code      int64
date             str
nav          float64
dtype: object

Sample:
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692

NAV <= 0: 0
Duplicates: 0
Missing: 0


In [7]:
# CLEAN nav_history

# 1. Fix date type
nav_history['date'] = pd.to_datetime(nav_history['date'])

# 2. Sort by fund and date
nav_history = nav_history.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# 3. Create a complete date range for each fund and forward fill gaps
all_dates = pd.date_range(nav_history['date'].min(), nav_history['date'].max(), freq='D')
all_codes = nav_history['amfi_code'].unique()

# Create a full grid of every fund x every date
full_index = pd.MultiIndex.from_product([all_codes, all_dates], names=['amfi_code', 'date'])
nav_full = nav_history.set_index(['amfi_code', 'date']).reindex(full_index).reset_index()

# Forward fill missing NAVs (weekends/holidays)
nav_full['nav'] = nav_full.groupby('amfi_code')['nav'].ffill()

# Drop any remaining NaN (before fund's launch date)
nav_clean = nav_full.dropna(subset=['nav'])

print("AFTER CLEANING:")
print("Original shape:", nav_history.shape)
print("After filling gaps:", nav_clean.shape)
print("Date dtype:", nav_clean['date'].dtype)
print("Missing:", nav_clean.isnull().sum().sum())
print("Sample:")
print(nav_clean.head())

AFTER CLEANING:
Original shape: (46000, 3)
After filling gaps: (64320, 3)
Date dtype: datetime64[us]
Missing: 0
Sample:
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [8]:
os.makedirs('../data/processed', exist_ok=True)
nav_clean.to_csv('../data/processed/02_nav_history_clean.csv', index=False)
print("✅ Saved 02_nav_history_clean.csv")
print(f"   Rows: {len(nav_clean)}")

✅ Saved 02_nav_history_clean.csv
   Rows: 64320


In [9]:
# ============================================
# CLEAN 08_investor_transactions.csv
# ============================================

print("BEFORE CLEANING:")
print("Shape:", transactions.shape)
print("\nDtypes:")
print(transactions.dtypes)
print("\nTransaction types:")
print(transactions['transaction_type'].value_counts())
print("\nKYC status:")
print(transactions['kyc_status'].value_counts())
print("\nAmount <= 0:", (transactions['amount_inr'] <= 0).sum())
print("Missing:", transactions.isnull().sum().sum())
print("Duplicates:", transactions.duplicated().sum())

BEFORE CLEANING:
Shape: (32778, 13)

Dtypes:
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object

Transaction types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Amount <= 0: 0
Missing: 0
Duplicates: 0


In [10]:
# CLEAN transactions

# 1. Fix date type
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

# 2. Check KYC status values
print("KYC status values:")
print(transactions['kyc_status'].value_counts())

# 3. Validate transaction types are only expected values
valid_types = ['SIP', 'Lumpsum', 'Redemption']
invalid_types = transactions[~transactions['transaction_type'].isin(valid_types)]
print(f"\nInvalid transaction types: {len(invalid_types)}")

# 4. Validate amount > 0
print(f"Invalid amounts (<=0): {(transactions['amount_inr'] <= 0).sum()}")

print("\nAFTER CLEANING:")
print("Date dtype:", transactions['transaction_date'].dtype)
print("Missing:", transactions.isnull().sum().sum())
print("Shape:", transactions.shape)

KYC status values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Invalid transaction types: 0
Invalid amounts (<=0): 0

AFTER CLEANING:
Date dtype: datetime64[us]
Missing: 0
Shape: (32778, 13)


In [12]:
transactions.to_csv('../data/processed/08_investor_transactions_clean.csv', index=False)
print("✅ Saved 08_investor_transactions_clean.csv")
print(f"   Rows: {len(transactions)}")

✅ Saved 08_investor_transactions_clean.csv
   Rows: 32778


In [13]:
# ============================================
# CLEAN 07_scheme_performance.csv
# ============================================

print("BEFORE CLEANING:")
print("Shape:", scheme_perf.shape)
print("\nDtypes:")
print(scheme_perf.dtypes)
print("\nExpense ratio range:")
print(f"  Min: {scheme_perf['expense_ratio_pct'].min()}")
print(f"  Max: {scheme_perf['expense_ratio_pct'].max()}")
print("\nInvalid expense ratios (outside 0.1-2.5%):", 
      ((scheme_perf['expense_ratio_pct'] < 0.1) | 
       (scheme_perf['expense_ratio_pct'] > 2.5)).sum())
print("\nReturn columns - any non-numeric:")
for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']:
    print(f"  {col}: {scheme_perf[col].dtype}")
print("\nMissing:", scheme_perf.isnull().sum().sum())
print("Duplicates:", scheme_perf.duplicated().sum())

BEFORE CLEANING:
Shape: (40, 19)

Dtypes:
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

Expense ratio range:
  Min: 0.55
  Max: 1.64

Invalid expense ratios (outside 0.1-2.5%): 0

Return columns - any non-numeric:
  return_1yr_pct: float64
  return_3yr_pct: float64
  return_5yr_pct: float64

Missing: 0
Duplicates: 0


In [14]:
# CLEAN scheme_performance

# 1. Flag funds with negative 1yr returns
scheme_perf['return_flag'] = scheme_perf['return_1yr_pct'].apply(
    lambda x: 'negative_return' if x < 0 else 'normal'
)

# 2. Flag invalid expense ratios just in case
scheme_perf['expense_ratio_flag'] = scheme_perf['expense_ratio_pct'].apply(
    lambda x: 'invalid' if x > 2.5 or x < 0.1 else 'valid'
)

print("AFTER CLEANING:")
print("Return flags:")
print(scheme_perf['return_flag'].value_counts())
print("\nExpense ratio flags:")
print(scheme_perf['expense_ratio_flag'].value_counts())
print("\nFunds with negative 1yr return:")
print(scheme_perf[scheme_perf['return_flag'] == 'negative_return'][['scheme_name', 'return_1yr_pct']])

AFTER CLEANING:
Return flags:
return_flag
normal    40
Name: count, dtype: int64

Expense ratio flags:
expense_ratio_flag
valid    40
Name: count, dtype: int64

Funds with negative 1yr return:
Empty DataFrame
Columns: [scheme_name, return_1yr_pct]
Index: []


In [15]:
scheme_perf.to_csv('../data/processed/07_scheme_performance_clean.csv', index=False)
print("✅ Saved 07_scheme_performance_clean.csv")
print(f"   Rows: {len(scheme_perf)}")

✅ Saved 07_scheme_performance_clean.csv
   Rows: 40


In [16]:
# Save remaining 7 files with date fixes

# fund_master
fund_master['launch_date'] = pd.to_datetime(fund_master['launch_date'])
fund_master.to_csv('../data/processed/01_fund_master_clean.csv', index=False)
print("✅ Saved 01_fund_master_clean.csv")

# aum_by_house
aum_by_house['date'] = pd.to_datetime(aum_by_house['date'])
aum_by_house.to_csv('../data/processed/03_aum_by_fund_house_clean.csv', index=False)
print("✅ Saved 03_aum_by_fund_house_clean.csv")

# monthly_sip
monthly_sip['month'] = pd.to_datetime(monthly_sip['month'])
monthly_sip.to_csv('../data/processed/04_monthly_sip_inflows_clean.csv', index=False)
print("✅ Saved 04_monthly_sip_inflows_clean.csv")

# category_inflows
category_inflows['month'] = pd.to_datetime(category_inflows['month'])
category_inflows.to_csv('../data/processed/05_category_inflows_clean.csv', index=False)
print("✅ Saved 05_category_inflows_clean.csv")

# folio_count
folio_count['month'] = pd.to_datetime(folio_count['month'])
folio_count.to_csv('../data/processed/06_industry_folio_count_clean.csv', index=False)
print("✅ Saved 06_industry_folio_count_clean.csv")

# portfolio
portfolio.to_csv('../data/processed/09_portfolio_holdings_clean.csv', index=False)
print("✅ Saved 09_portfolio_holdings_clean.csv")

# benchmark
benchmark['date'] = pd.to_datetime(benchmark['date'])
benchmark.to_csv('../data/processed/10_benchmark_indices_clean.csv', index=False)
print("✅ Saved 10_benchmark_indices_clean.csv")

✅ Saved 01_fund_master_clean.csv
✅ Saved 03_aum_by_fund_house_clean.csv
✅ Saved 04_monthly_sip_inflows_clean.csv
✅ Saved 05_category_inflows_clean.csv
✅ Saved 06_industry_folio_count_clean.csv
✅ Saved 09_portfolio_holdings_clean.csv
✅ Saved 10_benchmark_indices_clean.csv


In [17]:
from sqlalchemy import create_engine
import sqlite3

# Create SQLite database
engine = create_engine('sqlite:///../data/bluestock_mf.db')
print("✅ Database created!")
print("Location: data/bluestock_mf.db")

✅ Database created!
Location: data/bluestock_mf.db


In [18]:
import os
print(os.path.exists('../data/bluestock_mf.db'))
print(os.listdir('../data'))

False
['processed', 'raw']


In [19]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("CREATE TABLE IF NOT EXISTS test (id INTEGER)"))
    conn.commit()

print("File exists now:", os.path.exists('../data/bluestock_mf.db'))

File exists now: True


In [20]:
# Drop the test table
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS test"))
    conn.commit()

print("Test table removed")

Test table removed


In [21]:
schema_sql = """
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    risk_category TEXT
);

CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code INTEGER,
    date DATE,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr INTEGER,
    state TEXT,
    city TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code INTEGER,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    sharpe_ratio REAL,
    expense_ratio_pct REAL,
    aum_crore INTEGER,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with engine.connect() as conn:
    for statement in schema_sql.split(';'):
        if statement.strip():
            conn.execute(text(statement))
    conn.commit()

print("✅ Schema created!")

✅ Schema created!


In [22]:
# Load cleaned CSVs and prepare for database

fund_master_clean = pd.read_csv('../data/processed/01_fund_master_clean.csv')
nav_clean = pd.read_csv('../data/processed/02_nav_history_clean.csv')
transactions_clean = pd.read_csv('../data/processed/08_investor_transactions_clean.csv')
scheme_perf_clean = pd.read_csv('../data/processed/07_scheme_performance_clean.csv')

# Select only the columns we need for dim_fund
dim_fund_data = fund_master_clean[['amfi_code', 'fund_house', 'scheme_name', 'category', 
                                     'sub_category', 'plan', 'launch_date', 'benchmark', 
                                     'expense_ratio_pct', 'risk_category']]

# Select columns for fact_nav
fact_nav_data = nav_clean[['amfi_code', 'date', 'nav']]

# Select columns for fact_transactions
fact_transactions_data = transactions_clean[['investor_id', 'transaction_date', 'amfi_code', 
                                               'transaction_type', 'amount_inr', 'state', 
                                               'city', 'kyc_status']]

# Select columns for fact_performance
fact_performance_data = scheme_perf_clean[['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 
                                             'sharpe_ratio', 'expense_ratio_pct', 'aum_crore']]

print("Data prepared!")
print("dim_fund:", dim_fund_data.shape)
print("fact_nav:", fact_nav_data.shape)
print("fact_transactions:", fact_transactions_data.shape)
print("fact_performance:", fact_performance_data.shape)

Data prepared!
dim_fund: (40, 10)
fact_nav: (64320, 3)
fact_transactions: (32778, 8)
fact_performance: (40, 6)


In [23]:
# Load into database
dim_fund_data.to_sql('dim_fund', engine, if_exists='replace', index=False)
fact_nav_data.to_sql('fact_nav', engine, if_exists='replace', index=False)
fact_transactions_data.to_sql('fact_transactions', engine, if_exists='replace', index=False)
fact_performance_data.to_sql('fact_performance', engine, if_exists='replace', index=False)

print("✅ All tables loaded into database!")

✅ All tables loaded into database!


In [24]:
# Verify row counts match
with engine.connect() as conn:
    for table in ['dim_fund', 'fact_nav', 'fact_transactions', 'fact_performance']:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"{table}: {count} rows")

dim_fund: 40 rows
fact_nav: 64320 rows
fact_transactions: 32778 rows
fact_performance: 40 rows


In [25]:
query1 = """
SELECT scheme_name, fund_house, category, expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC;
"""

result = pd.read_sql(query1, engine)
print("Funds with expense ratio < 1%:")
print(result)

Funds with expense ratio < 1%:
                                          scheme_name  \
0   Nippon India Gilt Securities Fund - Regular - ...   
1        HDFC Short Term Debt Fund - Regular - Growth   
2                Kotak Liquid Fund - Regular - Growth   
3            SBI Bluechip Fund - Direct Plan - Growth   
4           SBI Small Cap Fund - Direct Plan - Growth   
5       Nippon India Large Cap Fund - Direct - Growth   
6            ICICI Pru Liquid Fund - Regular - Growth   
7                Axis Bluechip Fund - Direct - Growth   
8        SBI Magnum Gilt Fund - Regular Plan - Growth   
9   HDFC Mid-Cap Opportunities Fund - Direct - Growth   
10                ABSL Liquid Fund - Regular - Growth   
11          ICICI Pru Bluechip Fund - Direct - Growth   
12                     Nippon India ETF Nifty 50 BeES   
13           HDFC Top 100 Fund - Direct Plan - Growth   

                  fund_house category  expense_ratio_pct  
0            Nippon India MF     Debt               0.

In [26]:
query2 = """
SELECT fund_house, COUNT(*) as num_funds, AVG(expense_ratio_pct) as avg_expense_ratio
FROM dim_fund
GROUP BY fund_house
ORDER BY num_funds DESC;
"""

result = pd.read_sql(query2, engine)
print("Number of funds and avg expense ratio per fund house:")
print(result)

Number of funds and avg expense ratio per fund house:
                 fund_house  num_funds  avg_expense_ratio
0           SBI Mutual Fund          5           1.024000
1           Nippon India MF          5           1.040000
2       ICICI Prudential MF          5           1.146000
3          HDFC Mutual Fund          5           1.038000
4         Kotak Mahindra MF          4           1.300000
5          Axis Mutual Fund          4           1.287500
6           UTI Mutual Fund          3           1.573333
7            Mirae Asset MF          3           1.526667
8           DSP Mutual Fund          3           1.556667
9  Aditya Birla Sun Life MF          3           1.306667


In [27]:
query3 = """
SELECT f.scheme_name, f.fund_house, p.return_1yr_pct, p.sharpe_ratio
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
ORDER BY p.return_1yr_pct DESC
LIMIT 5;
"""

result = pd.read_sql(query3, engine)
print("Top 5 funds by 1-year return:")
print(result)

Top 5 funds by 1-year return:
                                      scheme_name                fund_house  \
0          ABSL Small Cap Fund - Regular - Growth  Aditya Birla Sun Life MF   
1      SBI Small Cap Fund - Regular Plan - Growth           SBI Mutual Fund   
2          Axis Small Cap Fund - Regular - Growth          Axis Mutual Fund   
3  Nippon India Small Cap Fund - Regular - Growth           Nippon India MF   
4       SBI Small Cap Fund - Direct Plan - Growth           SBI Mutual Fund   

   return_1yr_pct  sharpe_ratio  
0           24.93          0.90  
1           24.56          0.94  
2           21.97          0.84  
3           21.30          0.81  
4           20.59          0.93  


In [28]:
query4 = """
SELECT state, COUNT(*) as num_transactions, SUM(amount_inr) as total_amount
FROM fact_transactions
WHERE transaction_type = 'SIP'
GROUP BY state
ORDER BY total_amount DESC;
"""
print("Q4 - SIP transactions by state:")
print(pd.read_sql(query4, engine))

Q4 - SIP transactions by state:
             state  num_transactions  total_amount
0   Madhya Pradesh              1796      20682243
1           Punjab              1792      20140064
2        Telangana              1656      18620216
3       Tamil Nadu              1642      18404368
4          Gujarat              1653      18378904
5          Haryana              1690      18176696
6        Karnataka              1620      17696903
7    Uttar Pradesh              1625      17534858
8      West Bengal              1638      17495769
9            Delhi              1606      17113608
10       Rajasthan              1499      16544233
11     Maharashtra              1499      16445629


In [29]:
query5 = """
SELECT amfi_code, strftime('%Y-%m', date) as month, AVG(nav) as avg_nav
FROM fact_nav
GROUP BY amfi_code, month
ORDER BY amfi_code, month
LIMIT 20;
"""
print("\nQ5 - Average NAV per month (sample):")
print(pd.read_sql(query5, engine))


Q5 - Average NAV per month (sample):
    amfi_code    month     avg_nav
0      100016  2022-01  511.923007
1      100016  2022-02  514.538068
2      100016  2022-03  522.286481
3      100016  2022-04  526.114670
4      100016  2022-05  504.335713
5      100016  2022-06  465.377240
6      100016  2022-07  436.773094
7      100016  2022-08  420.921258
8      100016  2022-09  422.428063
9      100016  2022-10  431.346506
10     100016  2022-11  463.835560
11     100016  2022-12  480.873619
12     100016  2023-01  490.845510
13     100016  2023-02  492.446579
14     100016  2023-03  545.736081
15     100016  2023-04  567.037887
16     100016  2023-05  565.489110
17     100016  2023-06  570.709487
18     100016  2023-07  578.383042
19     100016  2023-08  569.720910


In [30]:
query6 = """
SELECT f.scheme_name, f.fund_house, p.aum_crore
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
"""
print("\nQ6 - Top 5 funds by AUM:")
print(pd.read_sql(query6, engine))


Q6 - Top 5 funds by AUM:
                                         scheme_name         fund_house  \
0  Mirae Asset Emerging Bluechip Fund - Regular -...     Mirae Asset MF   
1      Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF   
2     Nippon India Small Cap Fund - Regular - Growth    Nippon India MF   
3         DSP Top 100 Equity Fund - Regular - Growth    DSP Mutual Fund   
4                UTI Mid Cap Fund - Regular - Growth    UTI Mutual Fund   

   aum_crore  
0      49046  
1      47469  
2      43630  
3      41828  
4      41728  


In [31]:
query7 = """
SELECT kyc_status, transaction_type, COUNT(*) as count, SUM(amount_inr) as total_amount
FROM fact_transactions
GROUP BY kyc_status, transaction_type
ORDER BY kyc_status, total_amount DESC;
"""
print("\nQ7 - Transaction breakdown by KYC status:")
print(pd.read_sql(query7, engine))


Q7 - Transaction breakdown by KYC status:
  kyc_status transaction_type  count  total_amount
0    Pending          Lumpsum    669     176361972
1    Pending       Redemption    378      93419874
2    Pending              SIP   1585      17626203
3   Verified          Lumpsum   7426    1883459476
4   Verified       Redemption   4589    1151105617
5   Verified              SIP  18131     199607288


In [32]:
query8 = """
SELECT f.scheme_name, f.category, p.sharpe_ratio, p.return_3yr_pct
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
WHERE p.sharpe_ratio > 1
ORDER BY p.sharpe_ratio DESC;
"""
print("\nQ8 - Funds with Sharpe ratio > 1:")
print(pd.read_sql(query8, engine))


Q8 - Funds with Sharpe ratio > 1:
                                         scheme_name category  sharpe_ratio  \
0           ICICI Pru Liquid Fund - Regular - Growth     Debt          7.68   
1               Kotak Liquid Fund - Regular - Growth     Debt          6.18   
2                ABSL Liquid Fund - Regular - Growth     Debt          5.14   
3       HDFC Short Term Debt Fund - Regular - Growth     Debt          1.84   
4       SBI Magnum Gilt Fund - Regular Plan - Growth     Debt          1.52   
5  Nippon India Gilt Securities Fund - Regular - ...     Debt          1.33   
6          HDFC Top 100 Fund - Regular Plan - Growth   Equity          1.06   
7      Mirae Asset Large Cap Fund - Regular - Growth   Equity          1.06   
8          ICICI Pru Bluechip Fund - Direct - Growth   Equity          1.03   

   return_3yr_pct  
0            7.68  
1            6.18  
2            5.14  
3            7.37  
4            6.07  
5            5.31  
6           14.84  
7           14

In [34]:
result = pd.read_sql("PRAGMA table_info(fact_transactions);", engine)
print(result)

   cid              name    type  notnull dflt_value  pk
0    0       investor_id    TEXT        0       None   0
1    1  transaction_date    TEXT        0       None   0
2    2         amfi_code  BIGINT        0       None   0
3    3  transaction_type    TEXT        0       None   0
4    4        amount_inr  BIGINT        0       None   0
5    5             state    TEXT        0       None   0
6    6              city    TEXT        0       None   0
7    7        kyc_status    TEXT        0       None   0


In [35]:
# Re-select with age_group included
fact_transactions_data = transactions_clean[['investor_id', 'transaction_date', 'amfi_code', 
                                               'transaction_type', 'amount_inr', 'state', 
                                               'city', 'age_group', 'kyc_status']]

# Reload into database
fact_transactions_data.to_sql('fact_transactions', engine, if_exists='replace', index=False)
print("✅ fact_transactions reloaded with age_group column")

✅ fact_transactions reloaded with age_group column


In [36]:
query9 = """
SELECT age_group, transaction_type, AVG(amount_inr) as avg_amount, COUNT(*) as num_txns
FROM fact_transactions
GROUP BY age_group, transaction_type
ORDER BY age_group, avg_amount DESC;
"""
print("Q9 - Average transaction amount by age group:")
print(pd.read_sql(query9, engine))

Q9 - Average transaction amount by age group:
   age_group transaction_type     avg_amount  num_txns
0      18-25          Lumpsum  255328.409959      1205
1      18-25       Redemption  251532.867454       762
2      18-25              SIP   10953.073245      2949
3      26-35          Lumpsum  255718.703144      3372
4      26-35       Redemption  246907.993590      2028
5      26-35              SIP   10986.895696      8063
6      36-45       Redemption  256532.081190      1244
7      36-45          Lumpsum  252478.933198      1976
8      36-45              SIP   10885.758628      4926
9      46-55          Lumpsum  258080.536797       924
10     46-55       Redemption  254640.535135       555
11     46-55              SIP   11136.763478      2300
12       56+          Lumpsum  246767.733010       618
13       56+       Redemption  242530.764550       378
14       56+              SIP   11574.922192      1478


In [37]:
query10 = """
SELECT f.category, COUNT(DISTINCT f.amfi_code) as num_funds, 
       ROUND(AVG(p.return_1yr_pct), 2) as avg_1yr_return,
       ROUND(AVG(p.expense_ratio_pct), 2) as avg_expense_ratio
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
GROUP BY f.category
ORDER BY avg_1yr_return DESC;
"""
print("Q10 - Category-wise performance summary:")
print(pd.read_sql(query10, engine))

Q10 - Category-wise performance summary:
  category  num_funds  avg_1yr_return  avg_expense_ratio
0   Equity         34           15.80               1.34
1     Debt          6            6.28               0.67


In [38]:
print("Q4 - SIP by state:")
print(pd.read_sql(query4, engine))
print("\nQ6 - Top 5 by AUM:")
print(pd.read_sql(query6, engine))
print("\nQ8 - Sharpe ratio > 1:")
print(pd.read_sql(query8, engine))

Q4 - SIP by state:
             state  num_transactions  total_amount
0   Madhya Pradesh              1796      20682243
1           Punjab              1792      20140064
2        Telangana              1656      18620216
3       Tamil Nadu              1642      18404368
4          Gujarat              1653      18378904
5          Haryana              1690      18176696
6        Karnataka              1620      17696903
7    Uttar Pradesh              1625      17534858
8      West Bengal              1638      17495769
9            Delhi              1606      17113608
10       Rajasthan              1499      16544233
11     Maharashtra              1499      16445629

Q6 - Top 5 by AUM:
                                         scheme_name         fund_house  \
0  Mirae Asset Emerging Bluechip Fund - Regular -...     Mirae Asset MF   
1      Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF   
2     Nippon India Small Cap Fund - Regular - Growth    Nippon India MF  

In [39]:
queries_sql = """
-- Query 1: Funds with expense ratio below 1%
SELECT scheme_name, fund_house, category, expense_ratio_pct
FROM dim_fund
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC;

-- Query 2: Number of funds and avg expense ratio per fund house
SELECT fund_house, COUNT(*) as num_funds, AVG(expense_ratio_pct) as avg_expense_ratio
FROM dim_fund
GROUP BY fund_house
ORDER BY num_funds DESC;

-- Query 3: Top 5 funds by 1-year return
SELECT f.scheme_name, f.fund_house, p.return_1yr_pct, p.sharpe_ratio
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
ORDER BY p.return_1yr_pct DESC
LIMIT 5;

-- Query 4: SIP transactions by state
SELECT state, COUNT(*) as num_transactions, SUM(amount_inr) as total_amount
FROM fact_transactions
WHERE transaction_type = 'SIP'
GROUP BY state
ORDER BY total_amount DESC;

-- Query 5: Average NAV per month per fund
SELECT amfi_code, strftime('%Y-%m', date) as month, AVG(nav) as avg_nav
FROM fact_nav
GROUP BY amfi_code, month
ORDER BY amfi_code, month;

-- Query 6: Top 5 funds by AUM
SELECT f.scheme_name, f.fund_house, p.aum_crore
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;

-- Query 7: Transaction breakdown by KYC status
SELECT kyc_status, transaction_type, COUNT(*) as count, SUM(amount_inr) as total_amount
FROM fact_transactions
GROUP BY kyc_status, transaction_type
ORDER BY kyc_status, total_amount DESC;

-- Query 8: Funds with Sharpe ratio above 1
SELECT f.scheme_name, f.category, p.sharpe_ratio, p.return_3yr_pct
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
WHERE p.sharpe_ratio > 1
ORDER BY p.sharpe_ratio DESC;

-- Query 9: Average transaction amount by age group
SELECT age_group, transaction_type, AVG(amount_inr) as avg_amount, COUNT(*) as num_txns
FROM fact_transactions
GROUP BY age_group, transaction_type
ORDER BY age_group, avg_amount DESC;

-- Query 10: Category-wise fund count and average return
SELECT f.category, COUNT(DISTINCT f.amfi_code) as num_funds, 
       ROUND(AVG(p.return_1yr_pct), 2) as avg_1yr_return,
       ROUND(AVG(p.expense_ratio_pct), 2) as avg_expense_ratio
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
GROUP BY f.category
ORDER BY avg_1yr_return DESC;
"""

with open('../sql/queries.sql', 'w') as f:
    f.write(queries_sql)

print("✅ Saved queries.sql to sql/ folder")

✅ Saved queries.sql to sql/ folder


In [40]:
schema_sql_clean = """
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    risk_category TEXT
);

CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code INTEGER,
    date DATE,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr INTEGER,
    state TEXT,
    city TEXT,
    age_group TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code INTEGER,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    sharpe_ratio REAL,
    expense_ratio_pct REAL,
    aum_crore INTEGER,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with open('../sql/schema.sql', 'w') as f:
    f.write(schema_sql_clean)

print("✅ Saved schema.sql to sql/ folder")

✅ Saved schema.sql to sql/ folder


In [41]:
data_dictionary = """# Data Dictionary — Bluestock Mutual Fund Analytics

## dim_fund
Master table containing details of all 40 mutual fund schemes.

| Column | Type | Description |
|---|---|---|
| amfi_code | INTEGER (PK) | Unique AMFI scheme code identifying the fund |
| fund_house | TEXT | Asset Management Company managing the fund |
| scheme_name | TEXT | Official name of the fund scheme |
| category | TEXT | Equity / Debt / Hybrid |
| sub_category | TEXT | Large Cap / Mid Cap / Small Cap / Liquid etc. |
| plan | TEXT | Direct or Regular plan |
| launch_date | DATE | Date the fund was launched |
| benchmark | TEXT | Index the fund is benchmarked against |
| expense_ratio_pct | REAL | Annual fee charged by the fund (%) |
| risk_category | TEXT | SEBI risk classification |

## fact_nav
Daily Net Asset Value for each fund, including weekends/holidays (forward-filled).

| Column | Type | Description |
|---|---|---|
| amfi_code | INTEGER (FK) | References dim_fund |
| date | DATE | NAV date |
| nav | REAL | Net Asset Value in Rs. |

## fact_transactions
Investor transaction records — SIP, Lumpsum, and Redemption activity.

| Column | Type | Description |
|---|---|---|
| investor_id | TEXT | Unique investor identifier |
| transaction_date | DATE | Date of transaction |
| amfi_code | INTEGER (FK) | References dim_fund |
| transaction_type | TEXT | SIP / Lumpsum / Redemption |
| amount_inr | INTEGER | Transaction amount in Rupees |
| state | TEXT | Investor's state |
| city | TEXT | Investor's city |
| age_group | TEXT | Investor's age bracket |
| kyc_status | TEXT | Verified / Pending |

## fact_performance
Performance and risk metrics per fund.

| Column | Type | Description |
|---|---|---|
| amfi_code | INTEGER (FK) | References dim_fund |
| return_1yr_pct | REAL | 1-year absolute return (%) |
| return_3yr_pct | REAL | 3-year CAGR (%) |
| sharpe_ratio | REAL | Risk-adjusted return measure (higher is better) |
| expense_ratio_pct | REAL | Annual fee (%) |
| aum_crore | INTEGER | Assets Under Management in Rs. crore |

## Data Quality Notes

- nav_history expanded from 46,000 trading-day records to 64,320 calendar-day records via forward-fill for weekends/holidays
- All expense ratios validated within SEBI range (0.1% - 2.5%)
- KYC status standardised to two values: Verified, Pending
- No duplicate records found in source data
- yoy_growth_pct in monthly SIP data is null for 2022 (no prior year to compare against) — expected, not an error
"""

with open('../data_dictionary.md', 'w') as f:
    f.write(data_dictionary)

print("✅ Saved data_dictionary.md")

✅ Saved data_dictionary.md
